<a href="https://colab.research.google.com/github/Blackx16/contxt/blob/finetune%2Fgemma-gateway-270m/gemma-gateway-270m/finetune/train_gemma_gateway.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tune Gemma 3 270M → Crown-Jewels Gateway classifier

Runtime → **T4 GPU**. Trains on the ready-to-train chat data, exports a `q4f16` ONNX
(~273 MB) that runs on-device via Transformers.js **WASM**, evals base-vs-tuned, and
pushes to Hugging Face. See `docs/FINETUNE_PLAN.md`.


In [1]:
%pip -q install -U "transformers>=4.56" "trl>=0.12" peft datasets accelerate \
    onnx onnx_ir onnxruntime "optimum[onnxruntime]" huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.2/194.2 kB 8.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour

In [3]:
# Gemma is a GATED model. ONE-TIME: open https://huggingface.co/google/gemma-3-270m-it
# and click "Agree and access". Then create a token (Settings > Access Tokens, role
# 'Write') and paste it below. This same login is reused to push the model at the end.
from huggingface_hub import login
login()

In [4]:
# Get the contract (prompt.py) + committed data.
!git clone -q -b finetune/gemma-gateway-270m https://github.com/Blackx16/contxt.git /content/contxt || (cd /content/contxt && git pull -q)
import sys; sys.path.insert(0, "/content/contxt")
%cd /content/contxt
from finetune.prompt import INSTRUCTION, build_messages
print(INSTRUCTION[:120], "...")

/content/contxt
You are a privacy classifier for personal data in the Contxt Crown-Jewels Gateway. Decide the tier of ONE item.
- PRIVAT ...


In [5]:
import json, pathlib, datasets
def load(p):
    p = pathlib.Path(p)
    if not p.exists():
        from google.colab import files          # data not committed yet? upload it.
        up = files.upload(); p.write_bytes(next(iter(up.values())))
    return [json.loads(l) for l in p.read_text().splitlines() if l.strip()]
train = load("finetune/dataset/train.jsonl"); test = load("finetune/dataset/test.jsonl")
print(len(train), "train /", len(test), "test")
train_ds = datasets.Dataset.from_list(train)
print(train[0]["messages"][0]["content"][:200])

1438 train / 253 test
You are a privacy classifier for personal data in the Contxt Crown-Jewels Gateway. Decide the tier of ONE item.
- PRIVATE = crown jewels that must stay on the user's device: money amounts, finances (s


In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTTrainer, SFTConfig
BASE = "google/gemma-3-270m-it"
tok = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.bfloat16,
                                             attn_implementation="eager")
cfg = SFTConfig(
    output_dir="contxt-gw-270m", num_train_epochs=3, per_device_train_batch_size=16,
    gradient_accumulation_steps=1, learning_rate=5e-5, lr_scheduler_type="cosine",
    warmup_ratio=0.03, max_length=512, bf16=True, logging_steps=10,
    completion_only_loss=True,   # train on the JSON answer, not the repeated instruction
    save_strategy="epoch", report_to="none",
)
SFTTrainer(model=model, processing_class=tok, train_dataset=train_ds, args=cfg).train()
model.save_pretrained("contxt-gw-270m"); tok.save_pretrained("contxt-gw-270m")
# QLoRA fallback if VRAM is tight: load with load_in_4bit=True + LoraConfig(r=16, alpha=32,
# target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]),
# then model = model.merge_and_unload() before save_pretrained.

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

Tokenizing train dataset:   0%|          | 0/1438 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1438 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1438 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2, 'pad_token_id': 0}.


AttributeError: 'CausalLMOutputWithPast' object has no attribute 'last_hidden_state'

In [7]:
# Sanity: does it emit clean JSON now?
import torch
m = build_messages('Your HDFC loan EMI of Rs 54,000 is due on the 5th.')
ids = tok.apply_chat_template(m, add_generation_prompt=True, return_tensors="pt").to(model.device)
print(tok.decode(model.generate(ids, max_new_tokens=96, do_sample=False)[0][ids.shape[-1]:],
                 skip_special_tokens=True))

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
Caching is incompatible with gradient checkpointing in Gemma3DecoderLayer. Setting `past_key_values=None`.


```
**********************************************************************************************


In [ ]:
# Export to ONNX + quantize with Xenova's build_gemma.py (same script behind onnx-community).
!wget -q https://gist.githubusercontent.com/xenova/a219dbf3c7da7edd5dbb05f92410d7bd/raw/45f4c5a5227c1123efebe1e36d060672ee685a8e/build_gemma.py
!python build_gemma.py --model_name ./contxt-gw-270m --output ./onnx-out -p fp16 q4 q4f16
!ls -lah onnx-out/onnx

In [ ]:
# The ship gate: fine-tune vs base on the held-out set.
!python finetune/eval.py --model ./contxt-gw-270m --base google/gemma-3-270m-it \
    --test finetune/dataset/test.jsonl --out finetune/dataset/eval.md --backend hf
print(open("finetune/dataset/eval.md").read())

In [ ]:
# Push the ONNX folder to HF (reuses the login from the top of the notebook).
from huggingface_hub import upload_folder, create_repo
create_repo("Blackx16/contxt-gateway-270m-onnx", exist_ok=True)
upload_folder(folder_path="onnx-out", repo_id="Blackx16/contxt-gateway-270m-onnx",
              commit_message="fine-tuned gateway classifier (q4f16)")
print("pushed → Blackx16/contxt-gateway-270m-onnx")

## Deploy
In `extension/offscreen.js`:
```js
const MODEL_ID = 'Blackx16/contxt-gateway-270m-onnx';
const clf = await pipeline('text-generation', MODEL_ID, { dtype: 'q4f16', device: 'wasm' });
// call with build_messages(): clf([{role:'user', content: prompt}], ...) so the chat
// template is applied — matches training. NOT fp16/webgpu (onnxruntime#26732 garbage).
```
Then re-run `python server/verify_cha26.py` to confirm zero private leakage still holds.
